# 10 Beta Diversity and Age

We test whether fungal community composition is associated with age in years (`age_years_sample`). We compare whole-ASV and 16-mer Bray–Curtis distances using the same samples for both representations and both models. We adjust for sex and settlement type because their distributions may vary across the age range, and both may be associated with fungal community composition.

## 1. Setup

### 1.1 Imports

In [17]:
import pandas as pd
import qiime2 as q2
from skbio import DistanceMatrix
from qiime2 import Visualization

### 1.2 Notebook settings and paths

We define the input paths, the shared complete-case files, and the outputs for each feature representation and model.

In [30]:
# 1. Shared inputs and output directories
metadata_path = "../data/raw/Final_metadata_with_GHSL_exclude_other_sex.tsv"
kmer16_distance_path = "../data/processed_ignored/07_beta_diversity/kmer16_bray_curtis_distance.qza"
asv_distance_path = "../data/processed_ignored/07_beta_diversity/asv_bray_curtis_distance.qza"

out_dir = "../data/processed_ignored/10_beta_diversity_age"
vis_dir = "../data/processed/10_beta_diversity_age"

columns = ["Settlement_Type", "age_years_sample", "sex_sample"]


# 2. Complete-case inputs for both models
adjusted_age_out_dir = f"{out_dir}/adjusted_age"

adjusted_asv_age_distance_path = f"{adjusted_age_out_dir}/asv_bray_curtis_complete.qza"
adjusted_kmer_age_distance_path = f"{adjusted_age_out_dir}/kmer16_bray_curtis_complete.qza"

adjusted_age_metadata_path = f"{adjusted_age_out_dir}/metadata_complete.tsv"


# 3. Overall PERMANOVA outputs
unadjusted_permanova_dir = f"{vis_dir}/unadjusted_permanova"
adjusted_permanova_dir = f"{vis_dir}/adjusted_permanova"

asv_unadjusted_overall_qzv = f"{unadjusted_permanova_dir}/asv_overall.qzv"
kmer16_unadjusted_overall_qzv = f"{unadjusted_permanova_dir}/kmer16_overall.qzv"

asv_adjusted_overall_qzv = f"{adjusted_permanova_dir}/asv_overall.qzv"
kmer16_adjusted_overall_qzv = f"{adjusted_permanova_dir}/kmer16_overall.qzv"


# Create directories
!mkdir -p "$out_dir" "$vis_dir" "$adjusted_age_out_dir"
!mkdir -p "$unadjusted_permanova_dir" "$adjusted_permanova_dir"

### 1.3 Load inputs

We load the whole-ASV and 16-mer Bray–Curtis distance matrices from notebook 07 and the cleaned metadata from notebook 06.

In [31]:
metadata = q2.Metadata.load(metadata_path).to_dataframe()
distance_matrix = q2.Artifact.load(kmer16_distance_path).view(DistanceMatrix)
asv_matrix = q2.Artifact.load(asv_distance_path).view(DistanceMatrix)

print("Whole-ASV samples:", len(asv_matrix.ids))
print("16-mer samples:", len(distance_matrix.ids))
print("Metadata rows:", len(metadata))

Whole-ASV samples: 150
16-mer samples: 150
Metadata rows: 149


## 2. Select samples

### 2.1 Match sample IDs

We keep samples present in both Bray–Curtis matrices and in the metadata.

In [32]:
shared_ids = pd.Index(distance_matrix.ids).intersection(asv_matrix.ids)
shared_ids = shared_ids.intersection(metadata.index)

matched = metadata.loc[shared_ids, columns].copy()


print("Matched samples:", len(matched))


Matched samples: 149


### 2.2 Keep complete cases

We keep samples labeled Rural, Suburban, or Urban with a numeric age and recorded diet and sex. Both PERMANOVA models use this same subset.

In [33]:
matched["Settlement_Type"] = matched["Settlement_Type"].str.strip()
matched["sex_sample"] = matched["sex_sample"].str.strip().replace("", pd.NA)
matched["age_years_sample"] = pd.to_numeric(matched["age_years_sample"], errors="coerce")

age_complete = matched.dropna(subset=columns).copy()
complete_ids = age_complete.index.tolist()

print(age_complete.columns.tolist())
print("Complete-case samples:", len(complete_ids))

['Settlement_Type', 'age_years_sample', 'sex_sample']
Complete-case samples: 141


We subset both Bray–Curtis matrices to the complete-case sample IDs in the same order and save them with the matching metadata.

In [34]:
asv_distance_complete = asv_matrix.filter(complete_ids)
kmer16_distance_complete = distance_matrix.filter(complete_ids)

assert list(asv_distance_complete.ids) == complete_ids
assert list(kmer16_distance_complete.ids) == complete_ids

age_complete.index.name = "sample-id"
q2.Metadata(age_complete).save(adjusted_age_metadata_path)

q2.Artifact.import_data("DistanceMatrix", asv_distance_complete).save(
    adjusted_asv_age_distance_path
)
q2.Artifact.import_data("DistanceMatrix", kmer16_distance_complete).save(
    adjusted_kmer_age_distance_path
)

print("Saved both distance matrices and metadata for", len(complete_ids), "samples.")

Saved both distance matrices and metadata for 141 samples.


### 2.3 Summarize samples

We compare the number of samples with a numeric age before and after requiring complete metadata. We also summarize the age distribution among the complete cases.

In [35]:
age_samples = matched["age_years_sample"].dropna()

counts = pd.DataFrame({
    "With numeric age": [len(age_samples)],
    "With all four variables": [len(age_complete)],
}, index=["Samples"])

display(counts)
display(age_complete["age_years_sample"].describe())

,With numeric age,With all four variables
Samples,144,141


count    141.000000
mean      48.645390
std       18.723223
min        2.000000
25%       36.000000
50%       53.000000
75%       63.000000
max       82.000000
Name: age_years_sample, dtype: float64

## 3. Overall PERMANOVA

### 3.1 Unadjusted PERMANOVA

We test whether fungal community composition is associated with age in years. Both tests use the same 139 samples and include age as the only predictor.

#### 3.1.1 Whole ASVs

In [36]:
!qiime diversity adonis \
    --i-distance-matrix "$adjusted_asv_age_distance_path" \
    --m-metadata-file "$adjusted_age_metadata_path" \
    --p-formula "age_years_sample" \
    --o-visualization "$asv_unadjusted_overall_qzv"

Saved Visualization to: ../data/processed/10_beta_diversity_age/unadjusted_permanova/asv_overall.qzv


In [9]:
Visualization.load(asv_unadjusted_overall_qzv)

<visualization: Visualization uuid: eeed48a4-e36d-4e1b-b448-e22be6568b76>

#### 3.1.2 16-mers

In [37]:
!qiime diversity adonis \
    --i-distance-matrix "$adjusted_kmer_age_distance_path" \
    --m-metadata-file "$adjusted_age_metadata_path" \
    --p-formula "age_years_sample" \
    --o-visualization "$kmer16_unadjusted_overall_qzv"

Saved Visualization to: ../data/processed/10_beta_diversity_age/unadjusted_permanova/kmer16_overall.qzv


In [11]:
Visualization.load(kmer16_unadjusted_overall_qzv)

<visualization: Visualization uuid: 471fdff2-122a-408d-90a4-5297a626d0bc>

#### Unadjusted results

| **Age term** | **Whole ASVs** | **16-mers** |
| ------------ | -------------: | ----------: |
| Samples      |            141 |         141 |
| R²           |         0.0134 |      0.0190 |
| Pseudo-F     |          1.890 |       2.698 |
| p-value      |          **0.027** |       **0.004** |

Among the 141 samples included in both analyses, age explained 1.34% of the variation in fungal community composition for whole ASVs and 1.90% for 16-mers. Both p-values were below 0.05.


### 3.2 Adjusted PERMANOVA

We put age last in the formula to test whether fungal community composition is associated with age in years after accounting for settlement type, diet and sex. Both tests use the same 139 samples as the unadjusted tests.

#### 3.2.1 Whole ASVs

In [26]:
!qiime diversity adonis \
    --i-distance-matrix "$adjusted_asv_age_distance_path" \
    --m-metadata-file "$adjusted_age_metadata_path" \
    --p-formula "Settlement_Type + sex_sample + age_years_sample" \
    --o-visualization "$asv_adjusted_overall_qzv"

Saved Visualization to: ../data/processed/10_beta_diversity_age/adjusted_permanova/asv_overall.qzv


In [27]:
Visualization.load(asv_adjusted_overall_qzv)

<visualization: Visualization uuid: f5443ffe-7c6c-4aa2-ab5d-c17bb1b3ca44>

#### 3.2.2 16-mers

In [28]:
!qiime diversity adonis \
    --i-distance-matrix "$adjusted_kmer_age_distance_path" \
    --m-metadata-file "$adjusted_age_metadata_path" \
    --p-formula "Settlement_Type + sex_sample + age_years_sample" \
    --o-visualization "$kmer16_adjusted_overall_qzv"

Saved Visualization to: ../data/processed/10_beta_diversity_age/adjusted_permanova/kmer16_overall.qzv


In [29]:
Visualization.load(kmer16_adjusted_overall_qzv)

<visualization: Visualization uuid: dbc2180f-46cf-4173-bca7-ba56b6b3ba9b>

#### Adjusted results

| **Age term** | **Whole ASVs** | **16-mers** |
| ------------ | -------------: | ----------: |
| Samples      |            141 |         141 |
| R²           |         0.0142 |      0.0192 |
| Pseudo-F     |          2.029 |       2.741 |
| p-value      |          **0.025** |       **0.016** |

After accounting for settlement type and sex, age explained 1.42% of the variation in fungal community composition for whole ASVs and 1.92% for 16-mers. Both p-values were below 0.05.


### Compare unadjusted and adjusted overall PERMANOVA

All four tests used the same 141 samples. We compare the age term before and after accounting for settlement type and sex.

| **Bray–Curtis representation** | **Model**  | **Age R²** | **Pseudo-F** | **p-value** |
| ------------------------------ | ---------- | ---------: | -----------: | ----------: |
| Whole ASVs                     | Unadjusted |     0.0134 |        1.890 |       **0.027** |
| Whole ASVs                     | Adjusted   |     0.0142 |        2.029 |       **0.025** |
| 16-mers                        | Unadjusted |     0.0190 |        2.698 |       **0.004** |
| 16-mers                        | Adjusted   |     0.0192 |        2.741 |       **0.016** |

After adjustment, age R² increased slightly for whole ASVs and was nearly unchanged for 16-mers. All four p-values were below 0.05.


### Discussion

Age showed a small association with fungal community composition in both Bray–Curtis representations after accounting for settlement type and sex. Age explained less than 2% of the variation in every model, with consistently higher R² values for 16-mers than for whole ASVs.